In [ ]:
"""Inserts"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor

# To use the experimental IterativeImputer, we need to explicitly ask for it:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler

In [ ]:
"""Reading data"""
training_data=pd.read_parquet("data_parquet_2026/train.parquet")
sensors = pd.read_parquet("data_parquet_2026/sensors.parquet")
test_data = pd.read_parquet("data_parquet_2026/test.parquet")

In [ ]:
""" Ségrégation des profiles"""
n=int(len(training_data["power"])/len(training_data["sensor"].unique())/3) #27384/3 = 9128
power1 = training_data.query("sensor =='N0000'")
power2 = training_data.query("sensor =='N0000'")
power3 = training_data.query("sensor =='N0000'")

for senso in training_data["sensor"].unique():
    power1 = pd.concat((power1, training_data[training_data["sensor"] == senso][:n]))
    power2 = pd.concat((power2, training_data[training_data["sensor"] == senso][n+1:2*n]))
    power3 = pd.concat((power3, training_data[training_data["sensor"] == senso][2*n+1:]))

#power1.plot.scatter(x="time", y="power", alpha=0.5)
#power2.plot.scatter(x="time", y="power", alpha=0.5)
#power3.plot.scatter(x="time", y="power", alpha=0.5)

In [ ]:
training_data.info()
#training_data.head(5)
training_data["sensor"]
sensors["sensor"]
#test_data.sample(15)
#sensors.info()
#sensors.query("coor_z == 0") #tous =0
#sensors.head(21)




In [ ]:
#for sens in training_data["sensors"]:
 #   print(sens)
sampleP=training_data.query("time <= 1800000 and sensor=='N418'")
sampleP.head(10)

sampleP.plot.scatter(x="time", y="temperature", alpha=0.5)
#mean.plot.scatter(x="time", y="temperature", alpha=0.5)

In [ ]:
"""graphes tous pour 1 senseur"""

sampleN418 = power1.query(" sensor=='N418'")
#sampleN418.plot.scatter(x="time", y="power", alpha=0.5)
sampleN418.plot.scatter(x="time", y="temperature", alpha=0.5)
sensors.query("sensor == 'N418'")
sampleN418.head()

In [ ]:
b = training_data.query("time <= 10**9 and time >= 0.8*10**9")
b.plot.scatter(x="time", y="power", alpha=0.5)
b.plot.scatter(x="time", y="temperature", alpha=0.5)



In [ ]:
sampleN418b = sampleN418.sample(1000)
sampleN418b.plot.scatter(x="time", y="power", alpha=0.5)
sampleN418b.plot.scatter(x="time", y="temperature", alpha=0.5)


In [ ]:
"""Managing missing values"""
def managing_missing_val(sens : pd.DataFrame):
    sens = sens.reset_index(drop=True)
    temp = 0
    Nanlen = [] #si on veut faire mieux que la moyenne
    memo = []
    
    for i,val in enumerate(sens["temperature"]):
        if not val==val: #if nan
            memo.append(i)                
        else: #c'est pas un nan
            if type(temp)!=int: #si on a une valeur en mémoire
                if len(memo)!=0: #et qu'il y avait des nan avant
                    sens.loc[memo,"temperature"]=(temp+sens["temperature"][i])/2 #on remplace les nan par la moyenne de la mémoire et l'actuel
                    memo=[] #on oublie
                temp=sens["temperature"][i]
            else: #pas de valeur en mémoire
                if len(memo)!=0: #mais des nan avant
                    sens.loc[memo,"temperature"]=sens["temperature"][i] #on remplaces les nan par la valeur d'après
                    memo=[] #on oublie
        
    return sens

In [ ]:
sampleN418.isna().sum()
a=managing_missing_val(sampleN418)
a.isna().sum()